# Model-Tests

In [ ]:
%matplotlib inline
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

In [ ]:
data.head()

In [ ]:
#arr = data["Brand_Name"].unique()
#for i in arr:
#    print(i)

In [ ]:
df = pd.DataFrame(data)
df_apple = df.loc[df["Brand_Name"] == "google", ["Date", "Close", "Brand_Name"]]
display(df_apple)

In [ ]:
df_apple_preproc = df_apple.rename(columns={"Date": "timestamp", "Close": "target", "Brand_Name": "item_id"})
df_apple_preproc["item_id"] = df_apple_preproc['item_id'].astype("string")
df_apple_preproc["timestamp"] = df_apple_preproc['timestamp'].astype("string")
timecut = df_apple_preproc["timestamp"].str.slice(stop=10) #Cut hh:mm:ss and timezone
df_apple_preproc["timestamp"] = timecut
df_apple_preproc["timestamp"] = pd.to_datetime(timecut) #convert string into datetime64
df_apple_reordered =  df_apple_preproc[['item_id', 'timestamp', 'target']] #Reordering columns
df_irregular = TimeSeriesDataFrame(
    pd.DataFrame(df_apple_reordered)
)
df_regular = df_irregular.convert_frequency(freq="D")
df_filled = df_regular.fill_missing_values()
data = TimeSeriesDataFrame.from_data_frame(
    df = df_filled,
    id_column="item_id",
    timestamp_column="timestamp"
)

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

prediction_length = 10
train_data, test_data = data.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(prediction_length=prediction_length, freq="D").fit(
    train_data, presets="bolt_base", hyperparameters={"Chronos": {"fine_tune": True, "fine_tune_lr": 1e-5, "fine_tune_steps": 6000}},
    time_limit=60,
)

In [ ]:
predictions = predictor.predict(train_data)
predictor.plot(
    data=data,
    predictions=predictions,
    item_ids=data.item_ids[:2],
    max_history_length=200,
);